# One Hot Encoding 

One-hot encoding is a way to convert categorical variables into a numeric format that machine learning models can use, by representing each category as a binary vector with a single 1 and the rest 0s. It ensures the model doesn’t accidentally treat integer-coded categories as having an order or magnitude (e.g., that “3” > “1” in a meaningful way).

## What problem it solves
Many ML algorithms (linear models, neural nets, SVMs, etc.) expect numeric input and interpret numbers as having order and distance. If you have a categorical feature like:

color ∈ {"red", "green", "blue"}

city ∈ {"Delhi", "Mumbai", "Chennai"}

and you simply map them to integers (red→1, green→2, blue→3), the model might wrongly assume:

“blue” > “red”

“blue − green” has some meaning

One-hot encoding avoids this by giving each category its own binary column.

## How one-hot encoding works
For a categorical feature with k unique categories, one-hot encoding creates k new binary columns.

Each row has exactly one 1 (the “hot” bit) and the rest 0.

## Why it’s used in ML / NLP
Makes nominal (unordered) categories safe for models that assume numeric meaning.

### Common in:

- Traditional ML pipelines (with scikit-learn, etc.)

- Some NLP representations (e.g., one-hot per word in a tiny vocabulary, or per character)

- Input layers for neural networks when starting from categorical IDs

In **NLP** specifically, a pure one-hot representation of words leads to very high-dimensional, sparse vectors (dimension = vocabulary size), which is why embeddings (Word2Vec, GloVe, etc.) are often preferred for larger vocabularies.

## One-hot encoding vs label encoding
**Label encoding**: assign each category an integer (e.g., red→0, green→1, blue→2).

Pros: compact, no extra columns.

Cons: introduces an artificial order; bad for nominal features in linear models.

**One-hot encoding**: create one binary column per category.

Pros: no implied order; each category is orthogonal.

Cons: increases dimensionality, can lead to sparse data, especially with many categories.

## Rule of thumb:

Use one-hot for nominal features (no natural order): colors, cities, product types.

Use label encoding (or ordinal encoding) for ordinal features (with a natural order): ratings like low < medium < high, education levels, etc.

In [1]:
colors = ["red", "green", "blue", "red"]

In [2]:
## Build vocabulary
vocab = sorted(set(colors))
vocab

['blue', 'green', 'red']

In [4]:
color_to_index = {color: index for index, color in enumerate(vocab)}
color_to_index

{'blue': 0, 'green': 1, 'red': 2}

In [9]:
def one_hot_encode(category, vocab, mapping):
    vec = [0]*len(vocab)
    vec[mapping[category]] = 1
    return vec

one_hot_colors = [one_hot_encode(color, vocab, color_to_index) for color in colors]
print(f'Vocabulary: {vocab}')
print('Color One-Hot encoded:', one_hot_colors)

Vocabulary: ['blue', 'green', 'red']
Color One-Hot encoded: [[0, 0, 1], [0, 1, 0], [1, 0, 0], [0, 0, 1]]


In [10]:
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose"]

In [11]:
def one_hot_encode_2(list_categories):
    # Build vocabulary
    vocab = sorted(set(list_categories))
    # Create mapping from category to index
    mapping = {category: index for index, category in enumerate(vocab)}
    # Create one-hot encoded vectors
    one_hot_vector = []
    for category in list_categories:
        vec = [0]*len(vocab)
        vec[mapping[category]] = 1
        one_hot_vector.append(vec)
    return one_hot_vector

In [12]:
one_hot_cities = one_hot_encode_2(cities)

In [17]:
for i in range(len(cities)):
    print(f'{cities[i]}: {one_hot_cities[i]}')

New York: [0, 0, 0, 0, 1, 0, 0, 0, 0, 0]
Los Angeles: [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
Chicago: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Houston: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
Phoenix: [0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
Philadelphia: [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
San Antonio: [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
San Diego: [0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
Dallas: [0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
San Jose: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]


In real projects, we will usually use sklearn.preprocessing.OneHotEncoder or pandas’ get_dummies, but the concept is exactly this.

## Advantages of One-Hot Encoding
- No artificial order between categories

    - Unlike label encoding, OHE doesn’t imply that one category is “greater than” another.

    - Safe for nominal features like colors, cities, product types.

- Widely compatible with ML algorithms

    - Works well with linear models, logistic regression, SVMs, neural networks, and many tree-based models.

    - Many libraries and algorithms expect or handle binary/one-hot features naturally.

- Simple and interpretable

    - Easy to understand: each new column clearly means “is this category present?”

    - Debugging and feature inspection are straightforward.

- Avoids misleading numerical relationships

    - Prevents models from interpreting integer codes (0,1,2,…) as having magnitude or distance.

## Disadvantages of One-Hot Encoding
- High dimensionality (feature explosion)

    - For a feature with k unique categories, OHE creates k new columns.

    - With high-cardinality features (hundreds/thousands of categories), this becomes impractical.

- Sparse data matrices

    - Most entries are zeros; each row has only one 1 per encoded feature.

    - Some algorithms handle sparsity well, others become slower or less effective.

- Increased memory and computation cost

    - More columns → larger matrices → more memory usage and slower training/inference.

    - Can aggravate the **curse of dimensionality**, making it harder for models to generalize, especially with small datasets.

- Risk of Over-fitting with many categories

    - Many rare categories get their own features, which can lead to overfitting on noise.

    - Models may struggle to learn robust patterns when the feature space is huge and sparse.

- Not ideal for very high-cardinality or hierarchical categories

    - For things like user IDs, product IDs, ZIP codes, etc., OHE is usually a bad choice; embeddings, hashing, or target-based encodings are better.

### When to use vs avoid OHE
- Use OHE when:

    - The categorical feature has low to moderate cardinality (e.g., ≤ 10–50 unique values, depending on dataset size).

    - The feature is nominal (no natural order).

    - You’re using models that handle high-dimensional sparse data reasonably well (linear models, many tree-based models).

- Avoid or be cautious with OHE when:

    - The feature has high cardinality (hundreds/thousands of unique values).

    - Memory/computation is tight, or your dataset is already large.

    - You see many rare categories that could cause overfitting.